# Classification evaluation

Predict whether an approved order will arrive after its estimated delivery date. Training uses a 180-day historical window with a 45-day buffer. The notebook compares classifiers on a random stratified development split and evaluates monthly predictions using ranking, probability and daily-capacity measures. The months shown here were inspected during development, so their results are not an independent final test.

## 1. Setup and data

One row represents an approved order. Only information available by its scoring date may enter model features.

In [34]:
import os, sys
sys.path.insert(0, os.path.abspath('.') if os.path.isdir('src') else os.path.abspath('phase2'))

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

from src import LOOKBACK, PXD, RANDOM_STATE
from src.data import load_olist_data
from src.features import build_feature_table, NUM_COLS, CAT_COLS, LEAKAGE_COLS
from src.labels import get_required_dates, labels_as_of, attach_prediction_labels
from src.evaluation import classification_metrics
from classification_eval import extra_metrics, paired_monthly_ap

pd.set_option('display.max_columns', 30)
olist = load_olist_data(verbose=False)
orders = olist['orders']
features = build_feature_table(olist)
feature_cols = NUM_COLS + CAT_COLS
assert not set(feature_cols) & set(LEAKAGE_COLS)
assert features['order_id'].is_unique
print(f'{len(features):,} approved-order feature rows')

98,959 approved-order feature rows


## 2. Labels and the historical window

The 45-day gap gives delivery outcomes time to mature. `labels_as_of` counts an overdue, undelivered order as late; it excludes outcomes unavailable by the run date. We check label availability rather than assuming the gap resolves every order.

In [35]:
def training_window(run_date):
    start, end, _ = get_required_dates(run_date, lookback=LOOKBACK, pXd=PXD, verbose=False)
    rows = features.loc[features['order_approved_dt'].between(start, end)].copy()
    rows = labels_as_of(rows, run_date)
    known = rows.loc[rows['label_as_of_run'].notna()].copy()
    known['late'] = known['label_as_of_run'].astype(int)
    return rows, known

RUN_DATE = '2018-06-02'
window, labelled = training_window(RUN_DATE)
display(window['label_status'].value_counts().rename_axis('status').to_frame('orders'))
print(f'Known labels: {len(labelled):,}/{len(window):,} ({len(labelled)/len(window):.2%})')
assert labelled['order_approved_dt'].max() < pd.Timestamp(RUN_DATE) - pd.Timedelta(days=LOOKBACK)

,orders
status,
on_time_delivered,34593
late_delivered,4213
late_undelivered,1150
invalid_or_missing_dates,2
unresolved,2


Known labels: 39,956/39,960 (99.99%)


## 3. Random stratified development split

Reserve a stratified portion of the historical window for validation. Its results can guide model development; a later untouched sample is needed for final evaluation.

In [36]:
X_train, X_valid, y_train, y_valid = train_test_split(
    labelled[feature_cols], labelled['late'], test_size=30/PXD,
    random_state=RANDOM_STATE, stratify=labelled['late'])
print(f'Train: {len(X_train):,}; validation: {len(X_valid):,}; late rate: {y_valid.mean():.2%}')

Train: 33,296; validation: 6,660; late rate: 13.42%


## 4. Classification pipelines

Each pipeline fits preprocessing on its own training rows. Scores of 0.5 or above receive the late flag.

In [37]:
def prepare_catboost(X):
    data = X.copy()
    data[NUM_COLS] = data[NUM_COLS].astype(float)
    for column in CAT_COLS:
        data[column] = data[column].astype('string').fillna('Missing').astype(str)
    return data


def models_for(y):
    ratio = y.eq(0).sum() / y.eq(1).sum()
    plain = ColumnTransformer([
        ('numeric', SimpleImputer(strategy='median'), NUM_COLS),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), CAT_COLS),
    ], sparse_threshold=0)
    scaled = ColumnTransformer([
        ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')),
                              ('scaler', StandardScaler())]), NUM_COLS),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), CAT_COLS),
    ])
    return {
        'Logistic Regression': Pipeline([('preprocessor', scaled),
            ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000,
                                               random_state=RANDOM_STATE))]),
        'XGBoost': Pipeline([('preprocessor', clone(plain)),
            ('classifier', XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                                         scale_pos_weight=ratio, random_state=RANDOM_STATE,
                                         eval_metric='logloss'))]),
        'Random Forest': Pipeline([('preprocessor', clone(plain)),
            ('classifier', RandomForestClassifier(n_estimators=100, max_depth=10,
                                                   class_weight='balanced', random_state=RANDOM_STATE,
                                                   n_jobs=-1))]),
        'CatBoost': Pipeline([('preprocessor', FunctionTransformer(prepare_catboost,
                                                                   validate=False)),
            ('classifier', CatBoostClassifier(cat_features=CAT_COLS, iterations=100,
                                               scale_pos_weight=ratio, random_seed=RANDOM_STATE,
                                               verbose=False, allow_writing_files=False))]),
        'LightGBM': Pipeline([('preprocessor', clone(plain)),
            ('classifier', LGBMClassifier(scale_pos_weight=ratio, random_state=RANDOM_STATE,
                                          verbosity=-1, n_jobs=-1))]),
    }

models = models_for(y_train)
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f'Fitted {name}')

Fitted Logistic Regression
Fitted XGBoost
Fitted Random Forest
Fitted CatBoost
Fitted LightGBM


## 5. Development validation metrics

Accuracy measures overall correctness. Precision and recall describe the late-order decisions, while F1 combines them. ROC-AUC and average precision evaluate ranking across thresholds; average precision is especially useful for the less common late class. Brier score measures probability error (lower is better). The daily-capacity measure below assesses how many late orders a fixed contact budget could reach. No threshold tuning or probability calibration is fitted here.

In [38]:
validation_rows = labelled.loc[X_valid.index, ['order_id', 'order_approved_dt', 'late']].copy()
validation_rows = validation_rows.rename(columns={'order_approved_dt': 'approval_date',
                                                   'late': 'actual_label'})
validation_results = []
for name, model in models.items():
    score = model.predict_proba(X_valid)[:, list(model.classes_).index(1)]
    prediction = validation_rows.copy()
    prediction['score'] = score
    result = classification_metrics(y_valid, (score >= 0.5).astype(int), score)
    result.update(extra_metrics(prediction))
    validation_results.append({'model': name, **result})
validation_table = pd.DataFrame(validation_results).set_index('model')
display(validation_table[['accuracy', 'precision', 'recall', 'f1_score',
                          'roc_auc', 'average_precision', 'brier_score',
                          'coverage', 'lift']].round(3))

,accuracy,precision,recall,f1_score,roc_auc,average_precision,brier_score,coverage,lift
model,,,,,,,,,
Logistic Regression,0.688,0.228,0.554,0.323,0.666,0.254,0.222,0.242,2.180
XGBoost,0.742,0.283,0.604,0.386,0.744,0.398,0.179,0.284,2.563
Random Forest,0.725,0.269,0.609,0.373,0.729,0.374,0.193,0.273,2.462
CatBoost,0.726,0.269,0.607,0.373,0.728,0.381,0.182,0.266,2.402
LightGBM,0.738,0.282,0.616,0.387,0.742,0.396,0.178,0.277,2.503


## 6. Monthly development backtests

Each monthly backtest fits every model once on the second calendar day of the month, using the eligible historical window whose labels are mature by that run date. For example, the April 2 fit scores orders approved on April 1, then remains fixed for orders approved during the rest of April. The notebook calculates those daily scores in one batch after the fact; no April orders are used to fit the April model.

An order already delivered by the end of its approval day is not scored. Outcomes are attached at each order’s approval date plus 45 days. Unknown outcomes remain in the scored population and daily capacity; label-based metrics use only known outcomes and report their availability.

Each monthly fit uses the training portion of a random stratified split of its historical window.

In [39]:
def monthly_predictions(run_date):
    _, history = training_window(run_date)
    X_fit, _, y_fit, _ = train_test_split(
        history[feature_cols], history['late'], test_size=30/PXD,
        random_state=RANDOM_STATE, stratify=history['late'])
    month = pd.Period(run_date, freq='M')
    cohort = features.loc[features['order_approved_dt'].dt.to_period('M').eq(month)].copy()
    delivered_day = cohort['order_delivered_customer_date'].dt.normalize()
    cohort = cohort.loc[delivered_day.isna() |
                        delivered_day.gt(cohort['order_approved_dt'])].copy()
    if cohort.empty:
        return pd.DataFrame()
    base = cohort[['order_id', 'order_approved_dt']].rename(
        columns={'order_approved_dt': 'approval_date'})
    fitted = models_for(y_fit)
    scores = {}
    for name, model in fitted.items():
        model.fit(X_fit, y_fit)
        scores[name] = model.predict_proba(cohort[feature_cols])[:,
                          list(model.classes_).index(1)]
    scores['Equal-average ensemble'] = np.mean(list(scores.values()), axis=0)
    predictions = []
    for name, score in scores.items():
        frame = base.copy()
        frame['score'] = score
        frame['predicted_probability'] = score
        frame = attach_prediction_labels(frame, orders, lookback=LOOKBACK,
                                         prediction_threshold=0.5)
        frame['model'] = name
        frame['run_date'] = run_date
        predictions.append(frame.drop(columns=['predicted_probability']))
    return pd.concat(predictions, ignore_index=True)

# All three months were previously inspected, so they are development evidence.
RUN_DATES = ['2018-04-02', '2018-05-02', '2018-06-02']
monthly = pd.concat([monthly_predictions(day) for day in RUN_DATES], ignore_index=True)
assert not monthly.duplicated(['run_date', 'model', 'order_id']).any()
print(f'{monthly["order_id"].nunique():,} distinct scored orders')

19,963 distinct scored orders


## 7. Compare months and daily capacity

Precision and recall use the provisional 0.5 threshold. The 10% capacity is an illustrative contact budget, not a chosen business requirement. Coverage is the share of known late orders selected in the daily top 10%; lift compares that capture with random daily selection at the same capacity. Report label availability beside it.

In [40]:
monthly_results = []
for (run_date, name), group in monthly.groupby(['run_date', 'model'], sort=True):
    known = group.loc[group['actual_label'].notna()]
    y = known['actual_label'].astype(int)
    if known.empty:
        continue
    result = classification_metrics(y, (known['score'] >= 0.5).astype(int), known['score'])
    result.update(extra_metrics(group, fraction=0.10))
    monthly_results.append({'month': run_date[:7], 'model': name, **result})
monthly_results = pd.DataFrame(monthly_results)
display(monthly_results[['month', 'model', 'pr_auc', 'brier_score', 'precision',
                         'recall', 'f1_score', 'coverage', 'lift',
                         'label_availability']].round(3))
print('Mean across months (each month has equal weight):')
display(monthly_results.groupby('model')[['pr_auc', 'brier_score', 'precision',
                                           'recall', 'f1_score', 'coverage', 'lift',
                                           'label_availability']].mean().round(3))

,month,model,pr_auc,brier_score,precision,recall,f1_score,coverage,lift,label_availability
0,2018-04,CatBoost,0.117,0.234,0.099,0.600,0.170,0.193,1.892,1.000
1,2018-04,Equal-average ensemble,0.128,0.207,0.122,0.557,0.200,0.211,2.068,1.000
2,2018-04,LightGBM,0.129,0.224,0.109,0.591,0.184,0.211,2.068,1.000
3,2018-04,Logistic Regression,0.112,0.205,0.117,0.503,0.189,0.182,1.782,1.000
4,2018-04,Random Forest,0.140,0.181,0.153,0.476,0.232,0.218,2.134,1.000
5,2018-04,XGBoost,0.121,0.227,0.112,0.564,0.186,0.207,2.024,1.000
6,2018-05,CatBoost,0.147,0.332,0.095,0.717,0.167,0.168,1.647,1.000
7,2018-05,Equal-average ensemble,0.141,0.297,0.094,0.703,0.166,0.161,1.579,1.000
8,2018-05,LightGBM,0.139,0.341,0.091,0.762,0.162,0.173,1.698,1.000
9,2018-05,Logistic Regression,0.106,0.235,0.095,0.411,0.154,0.124,1.223,1.000


Mean across months (each month has equal weight):


,pr_auc,brier_score,precision,recall,f1_score,coverage,lift,label_availability
model,,,,,,,,
CatBoost,0.116,0.223,0.083,0.519,0.142,0.203,1.992,0.999
Equal-average ensemble,0.122,0.211,0.088,0.515,0.149,0.209,2.052,0.999
LightGBM,0.123,0.220,0.087,0.534,0.148,0.225,2.211,0.999
Logistic Regression,0.089,0.222,0.084,0.490,0.139,0.161,1.575,0.999
Random Forest,0.130,0.213,0.098,0.520,0.160,0.208,2.046,0.999
XGBoost,0.122,0.220,0.093,0.502,0.152,0.210,2.064,0.999


## 8. Paired model comparison

Each candidate is compared with the Logistic Regression baseline on the same orders. Resampling keeps whole dates together in seven-day blocks within each month, then averages the monthly AP differences. These descriptive intervals cover the development months and the fitted models; they do not include training variability or establish performance in a new month. No model is selected from these intervals alone.

In [41]:
comparison_rows = []
for candidate in ['XGBoost', 'Random Forest', 'CatBoost',
                  'LightGBM', 'Equal-average ensemble']:
    comparison_rows.append(paired_monthly_ap(monthly, candidate, repeats=499,
                                              block_days=7, seed=RANDOM_STATE))
display(pd.DataFrame(comparison_rows).round(3))

,candidate,reference,mean_ap_difference,ci_low,ci_high,months,block_days
0,XGBoost,LightGBM,-0.001,-0.005,0.004,3,7
1,Random Forest,LightGBM,0.008,0.002,0.014,3,7
2,CatBoost,LightGBM,-0.006,-0.014,0.000,3,7
3,LightGBM,LightGBM,0.000,0.000,0.000,3,7
4,Equal-average ensemble,LightGBM,-0.000,-0.006,0.004,3,7


## 9. Interpretation and limitations

The `pr_auc` column reports average precision. Compare monthly results and their variation before choosing a primary metric. The equal-average ensemble combines the five classifier scores with fixed weights; its outputs are not assumed calibrated probabilities.

The backtest months are development evidence. The analysis has not tuned thresholds, calibrated probabilities, evaluated additional feature groups or run an independent final test. Historical availability of some related-table features is an assumption because the data do not record revisions to those tables.

## References

- Brier, G. W. (1950). Verification of forecasts expressed in terms of probability. *Monthly Weather Review*, 78(1), 1-3. https://doi.org/10.1175/1520-0493(1950)078<0001:VOFEIT>2.0.CO;2
- Davis, J., & Goadrich, M. (2006). The relationship between Precision-Recall and ROC curves. *Proceedings of the 23rd International Conference on Machine Learning*, 233-240. https://doi.org/10.1145/1143844.1143874
- Saito, T., & Rehmsmeier, M. (2015). The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLoS ONE*, 10(3), e0118432. https://doi.org/10.1371/journal.pone.0118432
- Bergmeir, C., Hyndman, R. J., & Koo, B. (2018). A note on the validity of cross-validation for evaluating autoregressive time series prediction. *Data Mining and Knowledge Discovery*, 32, 1614-1630. https://doi.org/10.1007/s10618-018-0589-0
- Kunsch, H. R. (1989). The jackknife and the bootstrap for general stationary observations. *The Annals of Statistics*, 17(3), 1217-1241. https://doi.org/10.1214/aos/1176347265